In [1]:
import duckdb
import pandas as pd

In [2]:
con = duckdb.connect("../../data/research.duckdb")

In [3]:
PRICE_GLOB = "../../data/research-datasets/latent-fair-value/btc/price_snapshots/*.parquet"
CTX_GLOB = "../../data/research-datasets/latent-fair-value/btc/asset_context_snapshots/*.parquet"
OUT_PATH = "../../data/processed/btc_features_1m.parquet"

In [4]:
price_schema = con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{PRICE_GLOB}')
""").df()

In [5]:
ctx_schema = con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{CTX_GLOB}')
""").df()

In [6]:
price_schema, ctx_schema

(                           column_name column_type null   key default extra
 0                                   id      BIGINT  YES  None    None  None
 1                                asset     VARCHAR  YES  None    None  None
 2                measurement_timestamp      BIGINT  YES  None    None  None
 3                   filtered_timestamp      BIGINT  YES  None    None  None
 4                 observed_market_type     VARCHAR  YES  None    None  None
 5                       observed_asset     VARCHAR  YES  None    None  None
 6                   observed_bid_price      DOUBLE  YES  None    None  None
 7                    observed_bid_size      DOUBLE  YES  None    None  None
 8                   observed_ask_price      DOUBLE  YES  None    None  None
 9                    observed_ask_size      DOUBLE  YES  None    None  None
 10                  observed_mid_price      DOUBLE  YES  None    None  None
 11                 observed_microprice      DOUBLE  YES  None    None  None

In [7]:
price_preview = con.sql(f"""
    SELECT * FROM read_parquet('{PRICE_GLOB}') LIMIT 5
""").df()

In [8]:
ctx_preview = con.sql(f"""
    SELECT * FROM read_parquet('{CTX_GLOB}') LIMIT 5
""").df()

price_preview, ctx_preview

(   id asset  measurement_timestamp  filtered_timestamp observed_market_type  \
 0   1   BTC          1775141367369       1775141367369                 spot   
 1   2   BTC          1775141367369       1775141367369                 perp   
 2   3   BTC          1775141367474       1775141367474                 perp   
 3   4   BTC          1775141367608       1775141367608                 perp   
 4   5   BTC          1775141367665       1775141367665                 perp   
 
   observed_asset  observed_bid_price  observed_bid_size  observed_ask_price  \
 0            BTC             66790.0            0.02000             66796.0   
 1            BTC             66753.0            9.78675             66754.0   
 2            BTC             66753.0           25.75231             66754.0   
 3            BTC             66753.0           24.96010             66754.0   
 4            BTC             66753.0           17.19514             66754.0   
 
    observed_ask_size  ...  midprice

In [9]:
con.sql(f"""
    SELECT
        COUNT(*) AS n_rows
    FROM read_parquet('{PRICE_GLOB}')
""").df()

,n_rows
0,2932358


In [10]:
con.sql(f"""
    SELECT * FROM read_parquet('{PRICE_GLOB}') LIMIT 20
""").df()

,id,asset,measurement_timestamp,filtered_timestamp,observed_market_type,observed_asset,observed_bid_price,observed_bid_size,observed_ask_price,observed_ask_size,...,midprice_basis,microprice_1p5x_filtered_timestamp,microprice_1p5x_filtered_price,microprice_1p5x_basis,microprice_3x_filtered_timestamp,microprice_3x_filtered_price,microprice_3x_basis,recorded_at_ms,source_archive_key,source_etag
0,1,BTC,1775141367369,1775141367369,spot,BTC,66790.0,0.02000,66796.0,0.67261,...,-44.000000,1775141367369,66790.177363,-43.998141,1775141367369,66790.181457,-43.998143,1775141367708,latent-fair-value/btc/ubuntu-s-1vcpu-512mb-10g...,2f29d0579d93f02d9f98cc444ab41fb9
1,2,BTC,1775141367369,1775141367369,perp,BTC,66753.0,9.78675,66754.0,0.00040,...,-39.501113,1775141367369,66790.180204,-36.180284,1775141367369,66790.187124,-36.187218,1775141367720,latent-fair-value/btc/ubuntu-s-1vcpu-512mb-10g...,2f29d0579d93f02d9f98cc444ab41fb9
2,3,BTC,1775141367474,1775141367474,perp,BTC,66753.0,25.75231,66754.0,0.00040,...,-39.501097,1775141367474,66790.180222,-36.180238,1775141367474,66790.187146,-36.187163,1775141367808,latent-fair-value/btc/ubuntu-s-1vcpu-512mb-10g...,2f29d0579d93f02d9f98cc444ab41fb9
3,4,BTC,1775141367608,1775141367608,perp,BTC,66753.0,24.96010,66754.0,0.00040,...,-39.501097,1775141367608,66790.180222,-36.180238,1775141367608,66790.187146,-36.187162,1775141367813,latent-fair-value/btc/ubuntu-s-1vcpu-512mb-10g...,2f29d0579d93f02d9f98cc444ab41fb9
4,5,BTC,1775141367665,1775141367665,perp,BTC,66753.0,17.19514,66754.0,0.00040,...,-39.501097,1775141367665,66790.180220,-36.180243,1775141367665,66790.187144,-36.187167,1775141367923,latent-fair-value/btc/ubuntu-s-1vcpu-512mb-10g...,2f29d0579d93f02d9f98cc444ab41fb9
5,6,BTC,1775141367826,1775141367826,perp,BTC,66755.0,0.00303,66756.0,0.01093,...,-38.207899,1775141367826,66790.481024,-35.428796,1775141367826,66790.452079,-35.525328,1775141368163,latent-fair-value/btc/ubuntu-s-1vcpu-512mb-10g...,2f29d0579d93f02d9f98cc444ab41fb9
6,7,BTC,1775141367826,1775141367826,spot,BTC,66795.0,0.68261,66796.0,0.67261,...,-40.119403,1775141367826,66795.452127,-40.242932,1775141367826,66795.435964,-40.327631,1775141368176,latent-fair-value/btc/ubuntu-s-1vcpu-512mb-10g...,2f29d0579d93f02d9f98cc444ab41fb9
7,8,BTC,1775141367885,1775141367885,perp,BTC,66755.0,18.54100,66756.0,0.00944,...,-40.015952,1775141367885,66795.614866,-39.629106,1775141367885,66795.589490,-39.602889,1775141368208,latent-fair-value/btc/ubuntu-s-1vcpu-512mb-10g...,2f29d0579d93f02d9f98cc444ab41fb9
8,9,BTC,1775141367970,1775141367970,perp,BTC,66755.0,32.97071,66756.0,0.00794,...,-40.014139,1775141367970,66795.618786,-39.619255,1775141367970,66795.593167,-39.593622,1775141368348,latent-fair-value/btc/ubuntu-s-1vcpu-512mb-10g...,2f29d0579d93f02d9f98cc444ab41fb9
9,10,BTC,1775141368078,1775141368078,perp,BTC,66755.0,43.15109,66756.0,0.00794,...,-40.014110,1775141368078,66795.618867,-39.619054,1775141368078,66795.593244,-39.593431,1775141368358,latent-fair-value/btc/ubuntu-s-1vcpu-512mb-10g...,2f29d0579d93f02d9f98cc444ab41fb9


In [11]:
query = f"""
COPY (
WITH price_ranked AS (
    SELECT
        date_trunc('minute', to_timestamp(measurement_timestamp / 1000.0)) AS ts_min,
        to_timestamp(measurement_timestamp / 1000.0) AS price_ts,
        measurement_timestamp AS price_measurement_timestamp,
        spot_price,
        perp_price,
        spot_microprice,
        perp_microprice,
        basis,
        midprice_basis,
        microprice_1p5x_basis,
        microprice_3x_basis,
        row_number() OVER (
            PARTITION BY date_trunc('minute', to_timestamp(measurement_timestamp / 1000.0))
            ORDER BY measurement_timestamp DESC
        ) AS rn
    FROM read_parquet('{PRICE_GLOB}')
    WHERE lower(asset) = 'btc'
      AND spot_price IS NOT NULL
      AND perp_price IS NOT NULL
),

price_1m AS (
    SELECT
        ts_min,
        price_ts,
        price_measurement_timestamp,
        spot_price,
        perp_price,
        spot_microprice,
        perp_microprice,
        basis,
        midprice_basis,
        microprice_1p5x_basis,
        microprice_3x_basis
    FROM price_ranked
    WHERE rn = 1
),

ctx_clean AS (
    SELECT
        to_timestamp(recorded_at_ms / 1000.0) AS ctx_ts,
        recorded_at_ms,
        funding_rate,
        open_interest,
        oracle_price,
        mark_price,
        mid_price AS ctx_mid_price,
        premium,
        impact_bid_price,
        impact_ask_price,
        day_notional_volume
    FROM read_parquet('{CTX_GLOB}')
    WHERE lower(asset) = 'btc'
      AND lower(observed_market_type) = 'perp'
      AND recorded_at_ms IS NOT NULL
)

SELECT
    p.ts_min AS timestamp,
    p.price_measurement_timestamp,
    c.recorded_at_ms AS ctx_recorded_at_ms,
    p.spot_price,
    p.perp_price,
    p.spot_microprice,
    p.perp_microprice,
    p.basis AS basis_dollar,
    10000.0 * p.basis / NULLIF(p.spot_price, 0) AS basis_bps,
    p.midprice_basis,
    p.microprice_1p5x_basis,
    p.microprice_3x_basis,
    c.funding_rate,
    c.open_interest,
    c.oracle_price,
    c.mark_price,
    c.ctx_mid_price,
    c.premium,
    c.impact_bid_price,
    c.impact_ask_price,
    c.day_notional_volume
FROM price_1m p
ASOF LEFT JOIN ctx_clean c
    ON p.price_ts >= c.ctx_ts
ORDER BY timestamp
) TO '{OUT_PATH}' (FORMAT PARQUET);
"""

In [12]:
con.execute(query)
print("wrote:", OUT_PATH)

wrote: ../../data/processed/btc_features_1m.parquet


In [13]:
df = pd.read_parquet("../../data/processed/btc_features_1m.parquet")

In [14]:
df.tail()

,timestamp,price_measurement_timestamp,ctx_recorded_at_ms,spot_price,perp_price,spot_microprice,perp_microprice,basis_dollar,basis_bps,midprice_basis,...,microprice_3x_basis,funding_rate,open_interest,oracle_price,mark_price,ctx_mid_price,premium,impact_bid_price,impact_ask_price,day_notional_volume
5950,2026-04-06 18:01:00+00:00,1775498519984,1775498519652,69545.5,69478.5,69545.029200,69478.284853,-66.999785,-9.633950,-66.999785,...,-66.741117,-0.000013,28875.47874,69526.0,69483.0,69478.5,-0.000676,69478.0,69479.0,2.769821e+09
5951,2026-04-06 18:02:00+00:00,1775498579956,1775498579655,69533.5,69471.5,69533.176795,69471.000960,-61.990163,-8.915151,-61.990163,...,-62.150906,-0.000014,28879.04124,69516.0,69478.0,69472.5,-0.000619,69467.8,69473.0,2.770145e+09
5952,2026-04-06 18:03:00+00:00,1775498639969,1775498639064,69531.5,69476.5,69531.001993,69476.080448,-54.216709,-7.797431,-54.216709,...,-54.238287,-0.000014,28879.51816,69525.0,69481.0,69479.5,-0.000647,69479.0,69480.0,2.770401e+09
5953,2026-04-06 18:04:00+00:00,1775498699950,1775498699770,69521.5,69457.5,69521.871284,69457.925628,-64.291720,-9.247746,-64.291720,...,-64.244134,-0.000014,28888.31904,69498.0,69454.0,69456.5,-0.000554,69456.0,69459.5,2.773138e+09
5954,2026-04-06 18:05:00+00:00,1775498700616,1775498699770,69513.5,69457.5,69506.069526,69457.000190,-62.716872,-9.022258,-62.716872,...,-63.448752,-0.000014,28888.31904,69498.0,69454.0,69456.5,-0.000554,69456.0,69459.5,2.773138e+09


In [15]:
df.columns

Index(['timestamp', 'price_measurement_timestamp', 'ctx_recorded_at_ms',
       'spot_price', 'perp_price', 'spot_microprice', 'perp_microprice',
       'basis_dollar', 'basis_bps', 'midprice_basis', 'microprice_1p5x_basis',
       'microprice_3x_basis', 'funding_rate', 'open_interest', 'oracle_price',
       'mark_price', 'ctx_mid_price', 'premium', 'impact_bid_price',
       'impact_ask_price', 'day_notional_volume'],
      dtype='object')

In [16]:
df["open_interest"].isna().count()

5955

In [17]:
con.sql(f"""
WITH price_check AS (
    SELECT
        MIN(measurement_timestamp) AS min_ts,
        MAX(measurement_timestamp) AS max_ts
    FROM read_parquet('{PRICE_GLOB}')
),
ctx_check AS (
    SELECT
        MIN(measurement_timestamp) AS min_ts,
        MAX(measurement_timestamp) AS max_ts
    FROM read_parquet('{CTX_GLOB}')
)
SELECT
    'price' AS table_name,
    min_ts,
    max_ts
FROM price_check

UNION ALL

SELECT
    'context' AS table_name,
    min_ts,
    max_ts
FROM ctx_check
""").df()

,table_name,min_ts,max_ts
0,price,1775141367369,1775498700616
1,context,<NA>,<NA>


In [18]:
con.sql(f"""
WITH price_check AS (
    SELECT
        MIN(measurement_timestamp) AS min_ts,
        MAX(measurement_timestamp) AS max_ts
    FROM read_parquet('{PRICE_GLOB}')
),
ctx_check AS (
    SELECT
        MIN(measurement_timestamp) AS min_ts,
        MAX(measurement_timestamp) AS max_ts
    FROM read_parquet('{CTX_GLOB}')
)
SELECT
    'price' AS table_name,
    min_ts,
    max_ts
FROM price_check

UNION ALL

SELECT
    'context' AS table_name,
    min_ts,
    max_ts
FROM ctx_check
""").df()

,table_name,min_ts,max_ts
0,price,1775141367369,1775498700616
1,context,<NA>,<NA>


In [19]:
con.sql(f"""
SELECT
    measurement_timestamp,
    to_timestamp(measurement_timestamp) AS as_seconds,
    to_timestamp(measurement_timestamp / 1000.0) AS as_ms_wrong
FROM read_parquet('{CTX_GLOB}')
LIMIT 5
""").df()

,measurement_timestamp,as_seconds,as_ms_wrong
0,<NA>,NaT,NaT
1,<NA>,NaT,NaT
2,<NA>,NaT,NaT
3,<NA>,NaT,NaT
4,<NA>,NaT,NaT


In [20]:
con.sql(f"""
SELECT
    COUNT(*) AS n_rows,
    COUNT(measurement_timestamp) AS nonnull_measurement_timestamp,
    COUNT(recorded_at_ms) AS nonnull_recorded_at_ms,
    COUNT(funding_rate) AS nonnull_funding_rate,
    COUNT(open_interest) AS nonnull_open_interest
FROM read_parquet('{CTX_GLOB}')
""").df()

,n_rows,nonnull_measurement_timestamp,nonnull_recorded_at_ms,nonnull_funding_rate,nonnull_open_interest
0,348023,0,348023,348023,348023


In [21]:
con.sql(f"""
SELECT
    recorded_at_ms,
    to_timestamp(recorded_at_ms / 1000.0) AS ctx_ts,
    funding_rate,
    open_interest,
    mark_price,
    oracle_price
FROM read_parquet('{CTX_GLOB}')
WHERE lower(asset) = 'btc'
  AND lower(observed_market_type) = 'perp'
LIMIT 10
""").df()

,recorded_at_ms,ctx_ts,funding_rate,open_interest,mark_price,oracle_price
0,1775141367511,2026-04-02 10:49:27.511000-04:00,-0.00001,28432.31412,66740.0,66784.0
1,1775141368352,2026-04-02 10:49:28.352000-04:00,-0.00001,28432.31526,66740.0,66784.0
2,1775141369300,2026-04-02 10:49:29.300000-04:00,-0.00001,28432.41786,66740.0,66784.0
3,1775141370500,2026-04-02 10:49:30.500000-04:00,-0.00001,28432.41582,66755.0,66804.0
4,1775141371373,2026-04-02 10:49:31.373000-04:00,-0.00001,28432.58926,66755.0,66804.0
5,1775141372342,2026-04-02 10:49:32.342000-04:00,-0.00001,28432.60064,66755.0,66804.0
6,1775141373530,2026-04-02 10:49:33.530000-04:00,-0.00001,28432.82536,66784.0,66830.0
7,1775141374475,2026-04-02 10:49:34.475000-04:00,-0.00001,28432.86974,66784.0,66830.0
8,1775141375706,2026-04-02 10:49:35.706000-04:00,-0.00001,28432.88814,66784.0,66830.0
9,1775141376607,2026-04-02 10:49:36.607000-04:00,-0.00001,28432.91286,66783.0,66827.0


In [22]:
len(df), df["timestamp"].min(), df["timestamp"].max()

(5955,
 Timestamp('2026-04-02 14:49:00+0000', tz='UTC'),
 Timestamp('2026-04-06 18:05:00+0000', tz='UTC'))